# 01 — Data Generation

Generates the synthetic transit-event dataset (≈ 1.25 M rows, 90 days, 60 stations, 12 routes) and verifies schema and row count.

In [ ]:
import sys, os
sys.path.insert(0, '../src')
os.makedirs('../data', exist_ok=True)

import warnings
warnings.filterwarnings('ignore')


## 1 · Generate Dataset

In [ ]:
from data_gen import generate_all

generate_all('../data')
print("Generation complete.")


## 2 · Load & Inspect

In [ ]:
import pandas as pd

df = pd.read_parquet('../data/transit_events.parquet')
print(f"Shape : {df.shape}")
print(f"Columns ({len(df.columns)}):", df.columns.tolist())


In [ ]:
print("dtypes:")
print(df.dtypes)


In [ ]:
df.head()


## 3 · Verification

In [ ]:
TARGET_ROWS = 1_247_832
TOLERANCE   = 0.01          # ±1 %

actual = len(df)
assert abs(actual - TARGET_ROWS) / TARGET_ROWS < TOLERANCE, (
    f"Row count {actual} is more than 1 % away from {TARGET_ROWS}"
)
print(f"Row count OK: {actual:,}")

required_cols = [
    'event_id', 'timestamp', 'station_id', 'route_id',
    'scheduled_time', 'actual_time', 'delay_minutes',
    'weather_regime', 'hour_of_day', 'day_of_week',
    'next_hour_avg_delay',
]
missing = [c for c in required_cols if c not in df.columns]
assert not missing, f"Missing columns: {missing}"
print("Schema OK — all required columns present.")


## 4 · Target Rate

In [ ]:
df['y_primary'] = (df['next_hour_avg_delay'] >= 5.0).astype(int)

rate = df['y_primary'].mean()
print(f"Positive rate (next-hour delay ≥ 5 min): {rate:.3f}  ({rate*100:.1f} %)")
assert 0.10 < rate < 0.60, f"Unexpected positive rate: {rate:.3f}"


## 5 · Sample Distributions

In [ ]:
print("Numeric summary:")
df[['delay_minutes', 'next_hour_avg_delay', 'hour_of_day']].describe().round(3)


In [ ]:
print("Stations:", df['station_id'].nunique())
print("Routes  :", df['route_id'].nunique())
print("Days    :", df['timestamp'].dt.date.nunique())
print("\nWeather regime distribution:")
print(df['weather_regime'].value_counts())


## 6 · Generation Log

In [ ]:
log_path = '../data/GENERATION_LOG.md'
if os.path.exists(log_path):
    with open(log_path) as f:
        print(f.read())
else:
    print("No GENERATION_LOG.md found — check data_gen output.")
